# IceAnalytics

## 1- Carga y Exploración

In [2]:
import pandas as pd
import numpy as np

In [3]:
# El low memory evita warnings de tipos mixtos en columnas grandes
df_raw = pd.read_csv('product_activity.csv', low_memory=False)

In [4]:
# Vista rapida de los datos "crudos" y sus formatos 
print("=" * 70)
print("HEAD (primeras 5 filas)")
print("=" * 70)
print(df_raw.head())

HEAD (primeras 5 filas)
  user_id           created_at country plan_type  user_age   post_id  \
0  U01988  2025-02-18T02:07:44      PY       pro      26.0  P0008515   
1  U00236  2025-06-22T07:49:10      BR      free      27.0  P0001023   
2  U00791  2024-02-12T02:45:45      CL      free      28.0  P0003405   
3  U01522  2024-09-22T07:06:50      US      free      16.0  P0006524   
4  U01092  2025-07-18T02:27:52      PY      free       NaN  P0004665   

  post_category      post_created_at  votes_received  user_total_posts  \
0        sport   2025-05-07T20:55:28               7                16   
1          tech  2025-09-13T20:31:06               1                 9   
2          tech  2024-02-14T05:17:48              11                 2   
3       finance  2024-09-24T07:51:34               5                 2   
4     education  2025-07-24T04:56:56               7                 2   

   days_since_signup device_type  
0                 78      mobile  
1                 83        

In [5]:
print("\n" + "=" * 70)
print("INFO (tipos de dato y nulos por columna)")
print("=" * 70)

df_raw.info()


INFO (tipos de dato y nulos por columna)
<class 'pandas.DataFrame'>
RangeIndex: 8782 entries, 0 to 8781
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   user_id            8782 non-null   str    
 1   created_at         8782 non-null   str    
 2   country            8782 non-null   str    
 3   plan_type          8782 non-null   str    
 4   user_age           8028 non-null   float64
 5   post_id            8782 non-null   str    
 6   post_category      8782 non-null   str    
 7   post_created_at    8782 non-null   str    
 8   votes_received     8782 non-null   int64  
 9   user_total_posts   8782 non-null   int64  
 10  days_since_signup  8782 non-null   int64  
 11  device_type        8782 non-null   str    
dtypes: float64(1), int64(3), str(8)
memory usage: 823.4 KB


In [6]:
# Estadistica descriptiva de columnas numericas
# include='all' tambien nos muestra resumen de columnas categóricas
print("\n" + "=" * 70)
print("DESCRIBE (numéricas)")
print("=" * 70)
print(df_raw.describe())


DESCRIBE (numéricas)
          user_age  votes_received  user_total_posts  days_since_signup
count  8028.000000     8782.000000       8782.000000        8782.000000
mean     27.902591        6.918356          8.324186          29.479390
std       7.547052        5.127311          6.754906          36.819928
min      16.000000        0.000000          1.000000           0.000000
25%      22.000000        3.000000          4.000000           5.000000
50%      28.000000        6.000000          6.000000          17.000000
75%      33.000000        9.000000         11.000000          40.000000
max      58.000000       74.000000         39.000000         404.000000


In [7]:
# Descripcion de todas las columnas
print("\n" + "=" * 70)
print("DESCRIBE (todas las columnas, incluye categóricas)")
print("=" * 70)
print(df_raw.describe(include='all'))


DESCRIBE (todas las columnas, incluye categóricas)


       user_id           created_at country plan_type     user_age   post_id  \
count     8782                 8782    8782      8782  8028.000000      8782   
unique    2010                 2010      10        19          NaN      8610   
top     U00087  2024-01-08T12:18:18      US      free          NaN  P0002865   
freq        40                   40    1928      5978          NaN         2   
mean       NaN                  NaN     NaN       NaN    27.902591       NaN   
std        NaN                  NaN     NaN       NaN     7.547052       NaN   
min        NaN                  NaN     NaN       NaN    16.000000       NaN   
25%        NaN                  NaN     NaN       NaN    22.000000       NaN   
50%        NaN                  NaN     NaN       NaN    28.000000       NaN   
75%        NaN                  NaN     NaN       NaN    33.000000       NaN   
max        NaN                  NaN     NaN       NaN    58.000000       NaN   

       post_category      post_created_

In [8]:
# Dimensiones del dataset: (filas, columnas)
print(f"\nShape del dataset RAW: {df_raw.shape}")
print(f"Total de filas (eventos/post): {df_raw.shape[0]}")
print(f"Total de columnas: {df_raw.shape[1]}")



Shape del dataset RAW: (8782, 12)
Total de filas (eventos/post): 8782
Total de columnas: 12


## 2- Data Quality Report - Diagnostico

In [9]:
# Nulos por columna
print("=" * 60)
print("NULOS POR COLUMNA")
print("=" * 60)

nulos = df_raw.isnull().sum()
nulos_pct = (df_raw.isnull().mean() * 100).round(2)
reporte_nulos = pd.DataFrame({'nulos': nulos, '%nulos': nulos_pct})

print(reporte_nulos[reporte_nulos['nulos'] > 0])

NULOS POR COLUMNA
          nulos  %nulos
user_age    754    8.59


In [10]:
# Duplicados
print("\n" + "=" * 60)
print("DUPLICADOS")
print("=" * 60)

duplicados_exactos = df_raw.duplicated().sum()

print(f"Filas 100% duplicadas (todas las columnas iguales): {duplicados_exactos}")

# Duplicados por post_id
dup_post_id = df_raw['post_id'].duplicated().sum()
print(f"post_id repetidos: {dup_post_id}")
    


DUPLICADOS
Filas 100% duplicadas (todas las columnas iguales): 172
post_id repetidos: 172


In [11]:
# Valores únicos y frecuencias de columnas categóricas "sucias"
print("\n" + "=" * 60)
print("PLAN_TYPE — valores únicos y frecuencias")
print("=" * 60)

print(df_raw['plan_type'].value_counts(dropna=False))
#---------------------------------------------------------------
print("\n" + "=" * 60)
print("POST_CATEGORY — valores únicos y frecuencias")
print("=" * 60)

print(df_raw['post_category'].value_counts(dropna=False))
#---------------------------------------------------------------
print("\n" + "=" * 60)
print("DEVICE_TYPE — valores únicos y frecuencias")
print("=" * 60)

print(df_raw['device_type'].value_counts(dropna=False))


PLAN_TYPE — valores únicos y frecuencias
plan_type
free            5978
pro             1460
enterprise       306
Free             208
 free            197
FREE             196
FreE             189
PRo               46
 pro              44
PRO               44
Pro               38
Pro               35
EnterPrise        13
ENTERPRISE        11
 enterprise        7
Enterprise         7
premium            1
vip                1
enterprise+        1
Name: count, dtype: int64

POST_CATEGORY — valores únicos y frecuencias
post_category
tech           1187
life            913
sports          899
science         753
finance         739
gaming          727
music           614
health          601
education       592
travel          445
 tech            70
Tech             67
TECH             56
tehc             50
Life             46
Finance          44
 sport           42
sciense          42
gamming          40
 life            38
SPORTS           37
 finance         37
LIFE             36
Spo

In [12]:
# Chequeos lógicos de fechas (AÚN COMO STRING)
_created_at_temp = pd.to_datetime(df_raw['created_at'], errors='coerce')
_post_created_at_temp = pd.to_datetime(df_raw['post_created_at'], errors='coerce')

no_parseables_signup = _created_at_temp.isna().sum()
no_parseables_post = _post_created_at_temp.isna().sum()

print("\n" + "=" * 70)
print("FECHAS NO PARSEABLES")
print("=" * 70)
print(f"created_at (signup) no parseables: {no_parseables_signup}")
print(f"post_created_at no parseables: {no_parseables_post}")

post_antes_signup = (_post_created_at_temp < _created_at_temp).sum()
print(f"\nPosts creados ANTES del signup del usuario: {post_antes_signup}")


FECHAS NO PARSEABLES
created_at (signup) no parseables: 1
post_created_at no parseables: 1

Posts creados ANTES del signup del usuario: 100


In [13]:
# Chequeo de la columna days_since_signup
dias_calculado_temp = (_post_created_at_temp - _created_at_temp).dt.days
mismatch_dias = (dias_calculado_temp != df_raw['days_since_signup']).sum()

print(f"\nFilas donde days_since_signup (original) NO coincide con el cálculo real: {mismatch_dias}")


Filas donde days_since_signup (original) NO coincide con el cálculo real: 4479


### 3- Normalización + Recálculo + Quarantine 

In [14]:
df = df_raw.copy()

# Normalizar columnas con strings
for col in ['plan_type', 'post_category', 'device_type']:
    df[col] = df[col].astype(str).str.strip().str.lower()
    
# Diccionario canonico
PLAN_CANONICO = {'free', 'pro', 'enterprise'}
DEVICE_CANONICO = {'web', 'mobile', 'desktop'}
CATEGORY_CANONICO = {'tech', 'life', 'sports', 'science', 'finance', 'gaming', 'music', 'health', 'education', 'travel'}

# Correccion de errores de tipeo en post_category
TYPO_MAP_CATEGORY = {
    'tehc'      :   'tech',
    'sciense'   :   'science',
    'gamming'   :   'gaming',
    'sporst'    :   'sports',
    'sp0rts'    :   'sports',
    'sport'     :   'sports',
    'finanse'   :   'finance',
    'educatoin' :   'education',
    'healt'     :   'health',
    'lfe'       :   'life',
    'musc'      :   'music',
    'trvael'    :   'travel'
}

df['post_category'] = df['post_category'].replace(TYPO_MAP_CATEGORY)

# Los que no se encuentran en el diccionario(Quarantine)
flag_plan_invalido = ~df['plan_type'].isin(PLAN_CANONICO)
flag_device_invalido = ~df['device_type'].isin(DEVICE_CANONICO)
flag_category_invalida = ~df['post_category'].isin(CATEGORY_CANONICO)

print(f"plan_type fuera del diccionario: {flag_plan_invalido.sum()} ({df.loc[flag_plan_invalido, 'plan_type'].unique()})")
print(f"device_type fuera del diccionario: {flag_device_invalido.sum()} ({df.loc[flag_device_invalido, 'plan_type'].unique()})")
print(f"post_category fuera del diccionario: {flag_category_invalida.sum()} ({df.loc[flag_category_invalida, 'plan_type'].unique()})")


plan_type fuera del diccionario: 3 (<StringArray>
['premium', 'vip', 'enterprise+']
Length: 3, dtype: str)
device_type fuera del diccionario: 3 (<StringArray>
['free', 'vip']
Length: 2, dtype: str)
post_category fuera del diccionario: 2 (<StringArray>
['pro', 'vip']
Length: 2, dtype: str)


In [15]:
# Parseo de fechas a datatime
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df['post_created_at'] = pd.to_datetime(df['post_created_at'], errors='coerce')

flag_fecha_no_parseable = df['created_at'].isna() | df['post_created_at'].isna()

# Recalculo de 'days_since_signup'
df['days_since_signup_calc'] = (df['post_created_at'] - df['created_at']).dt.days

flag_post_antes_signup = df['days_since_signup_calc'] < 0

# Duplicados exactos
flag_duplicado = df.duplicated(keep='first')

# Construccion del reason_code
def construir_reason_code(row_idx):
    razones = []
    if flag_duplicado[row_idx]:
        razones.append('duplicated_row')
    if flag_fecha_no_parseable[row_idx]:
        razones.append('unparseable_date')
    if flag_post_antes_signup[row_idx]:
        razones.append('post_before_signup')
    if flag_plan_invalido[row_idx]:
        razones.append('invalid_plan_type')
    if flag_device_invalido[row_idx]:
        razones.append('invalid_device_type')
    if flag_category_invalida[row_idx]:
        razones.append('invalid_category')
    
    return '|'.join(razones)

mask_quarantine = (flag_duplicado | flag_fecha_no_parseable | flag_post_antes_signup | flag_plan_invalido | flag_device_invalido | flag_category_invalida)

df['reason_code'] = ''
df.loc[mask_quarantine, 'reason_code'] = [ construir_reason_code(i) for i in df.index[mask_quarantine]]

In [16]:
# CORE vs QUARANTINE
df_quarantine = df[mask_quarantine].copy()
df_core = df[~mask_quarantine].copy().drop(columns=['reason_code'])

print("\n" + "=" * 70)
print("RESUMEN DEL SPLIT")
print("=" * 60)

print(f"RAW total:        {len(df)}")
print(f"CORE (limpio):    {len(df_core)}")
print(f"QUARANTINE:       {len(df_quarantine)}")
print(f"% Quarantine:     {len(df_quarantine)/len(df)*100:.2f}%")

print("\nDesglose de reason_code en Quarantine:")
print(df_quarantine['reason_code'].value_counts())


RESUMEN DEL SPLIT
RAW total:        8782
CORE (limpio):    8507
QUARANTINE:       275
% Quarantine:     3.13%

Desglose de reason_code en Quarantine:
reason_code
duplicated_row                                            167
post_before_signup                                         95
duplicated_row|post_before_signup                           5
unparseable_date                                            2
invalid_plan_type                                           2
invalid_device_type                                         2
invalid_category                                            1
invalid_plan_type|invalid_device_type|invalid_category      1
Name: count, dtype: int64


## 4- Data Quality Report(resumen final)

In [17]:
dqr = {
    'filas_raw'                     : len(df_raw),
    'filas_core'                    : len(df_core),
    'filas_quarantine'              : len(df_quarantine),
    'pct_quarantine'                : round(len(df_quarantine) / len(df_raw) * 100, 2),
    'duplicados_removidos'          : int(flag_duplicado.sum()),
    'fechas_no_parseables'          : int(flag_fecha_no_parseable.sum()),
    'post_antes_signup'             : int(flag_post_antes_signup.sum()),
    'mismatch_days_since_signup'    : int(mismatch_dias),
    'pct_mismatch_dias'             : round(mismatch_dias / len(df_raw) * 100, 2)
}

for k, v in dqr.items():
    print(f"{k}: {v}")

filas_raw: 8782
filas_core: 8507
filas_quarantine: 275
pct_quarantine: 3.13
duplicados_removidos: 172
fechas_no_parseables: 2
post_antes_signup: 100
mismatch_days_since_signup: 4479
pct_mismatch_dias: 51.0


In [18]:
# Distribuciones de Volumen
print("Usuarios únicos por plan")

usuarios_por_plan = df_core.drop_duplicates('user_id').groupby('plan_type')['user_id'].count()
print(usuarios_por_plan)

print("--- Actividad (#posts) por país ---")
print(df_core.groupby('country')['post_id'].count().sort_values(ascending=False))

print("--- Actividad (#posts) por categoría ---")
print(df_core['post_category'].value_counts())

print("--- Actividad (#posts) por dispositivo ---")
print(df_core['device_type'].value_counts())


Usuarios únicos por plan
plan_type
enterprise      82
free          1545
pro            367
Name: user_id, dtype: int64
--- Actividad (#posts) por país ---
country
US    1873
BR    1600
AR    1168
PY     837
MX     802
CL     552
ES     546
CO     488
PE     393
UY     248
Name: post_id, dtype: int64
--- Actividad (#posts) por categoría ---
post_category
tech         1381
life         1034
sports       1017
science       870
finance       853
gaming        833
education     681
health        673
music         673
travel        492
Name: count, dtype: int64
--- Actividad (#posts) por dispositivo ---
device_type
web        4270
mobile     3656
desktop     581
Name: count, dtype: int64


In [19]:
# Engagement - Votos
print("--- Votos por plan: media, mediana, percentiles ---")
engagement_por_plan = df_core.groupby('plan_type')['votes_received'].agg(
    media='mean',
    mediana='median',
    p25=lambda x: x.quantile(0.25),
    p75=lambda x: x.quantile(0.75),
    std='std',
    n='count'
)

print(engagement_por_plan)

print("\n--- Votos por categoria ---")
print(df_core.groupby('post_category')['votes_received'].agg(['mean', 'median', 'count']))

print("\n--- Votos por dispositivo ---")
print(df_core.groupby('device_type')['votes_received'].agg(['mean', 'median', 'count']))

--- Votos por plan: media, mediana, percentiles ---
               media  mediana  p25   p75       std     n
plan_type                                               
enterprise  7.570571      7.0  4.0  10.0  4.897624   333
free        6.692706      6.0  3.0   9.0  5.029064  6567
pro         7.654014      7.0  4.0  10.0  5.490376  1607

--- Votos por categoria ---
                   mean  median  count
post_category                         
education      6.791483     6.0    681
finance        6.978898     6.0    853
gaming         7.186074     6.0    833
health         6.456166     6.0    673
life           6.173114     5.0   1034
music          6.286776     5.0    673
science        7.544828     6.0    870
sports         6.573255     6.0   1017
tech           7.687183     7.0   1381
travel         6.878049     6.0    492

--- Votos por dispositivo ---
                 mean  median  count
device_type                         
desktop      7.032702     6.0    581
mobile       6.828228   

In [20]:
# Evento vs Usuario
votos_promedio_evento = df_core['votes_received'].mean()

votos_promedio_por_usuario_individual = df_core.groupby('user_id')['votes_received'].mean()
votos_promedio_usuario = votos_promedio_por_usuario_individual.mean()

print(f"Promedio de votos POR EVENTO (post): {votos_promedio_evento:.2f}")
print(f"Promedio de votos POR USUARIO (agrupado): {votos_promedio_usuario:.2f}")
print(f"Diferencia: {votos_promedio_evento - votos_promedio_usuario:.2f}")

# Promedio de post por usuario
post_por_usuario = df_core.groupby('user_id')['post_id'].count()
print(f"\nPosts por usuario -> media {post_por_usuario.mean():.2f}, mediana: {post_por_usuario.median():.1f}, max: {post_por_usuario.max()}")

Promedio de votos POR EVENTO (post): 6.91
Promedio de votos POR USUARIO (agrupado): 6.90
Diferencia: 0.01

Posts por usuario -> media 4.27, mediana: 3.0, max: 39


In [ ]:
# Concentración - Regla de Pareto (Top 1%)

# Paso 1: Agregar actividad y votos totales POR USUARIO
actividad_usuario = df_core.groupby('user_id').agg(
    total_posts=('post_id', 'count'),
    total_votos=('votes_received', 'sum')
).reset_index()

n_usuarios = len(actividad_usuario)
top_1_pct_n = max(1, int(np.ceil(n_usuarios * 0.01)))

print(f"Total de usuarios únicos: {n_usuarios}")
print(f"Usuarios en el top 1%: {top_1_pct_n}")

# Paso 2: % de POSTS 

Total de usuarios únicos: 1994
Usuarios en el top 1%: 20
